## minGPT

Here we start getting into minGPT. We will perform basic setup and run a few experiments. 

We formulate a hypothesis:
Laplace estimation for binary IID data approaches entropy as it sees more of a sequence. We hypothesize that this laplace estimation can be done offline by training the transformer to estimate "true p" that a laplace estimation scheme would produce. This offline learning can therefore produce gains for small n's, even for IID data that has no "structure to exploit"


In [30]:
import sys
sys.path.append('../')

from mingpt.model import GPT
from mingpt.trainer import Trainer
import torch
from torch.utils.data import Dataset

In [31]:
# Universal variables

seq_samples = 500
num_samples = 10000
iters = 1000
learn_rate = 1e-4
model_save_folder = '../models/'

In [32]:

# We generate our dataset

class ModelDataset(Dataset):
    """
    IID binary sequences 
    """

    def __init__(self, p, n=10, num_samples=10000):
        self.n = n
        self.num_samples = num_samples 
        self.p = p

    def _sample(self):
        # sample probability p from uniform (0, 1)
        if callable(self.p):
            p = self.p()
        elif isinstance(self.p, list):
            p = self.p[torch.randint(len(self.p), (1,)).item()]
        else:
            p = self.p
        # generate IID Bernoulli sequence of length self.n
        seq = torch.bernoulli(torch.full((self.n,), p)).long()

        x = seq[:-1].clone()
        y = seq[1:].clone()

        return x, y

    def get_block_size(self):
        return self.n - 1

    def __getitem__(self, idx):
        return self._sample() 
    
    def __len__(self):
        return self.num_samples 


class FixedModelDataset(Dataset):
    def __init__(self, p, n=10, num_samples=10000):
        self.n = n
        self.block_size = n - 1
        # pre-generate all sequences at init
        self.data = []
        for _ in range(num_samples):
            seq = torch.bernoulli(torch.full((n,), p)).long()
            x = seq[:-1].clone()
            y = seq[1:].clone()
            self.data.append((x, y))

    def get_block_size(self):
        return self.block_size

    def __getitem__(self, idx):
        return self.data[idx]  # returns fixed pre-generated sequence

    def __len__(self):
        return len(self.data)
        



In [33]:
# Model configure

def model_configure(dataset, iters=1000, learn_rate=1e-4):
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = dataset.get_block_size()
    model = GPT(model_config)

    device = 'mps'
    model = model.to(device)

    train_config = Trainer.get_default_config()
    train_config.learning_rate = learn_rate # the model we're using is so small that we can go a bit faster
    train_config.max_iters = iters 
    train_config.num_workers = 0
    train_config.device = 'mps'
    trainer = Trainer(train_config, model, dataset)
    return model, trainer

def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_num}; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")

def train_run(model, trainer, output_name):
    save_dir = model_save_folder + output_name
    print(f'Saving to {save_dir}')
    trainer.set_callback('on_batch_end', batch_end_callback)
    trainer.run()
    torch.save({
        'state_dict': model.state_dict(),
        'block_size': model.block_size
    }, save_dir)
    print(f'Saved to {save_dir}')

# Model for IID

In [1]:
# Define model and trainer

p_iid_dataset = FixedModelDataset(p=0.3, n=seq_samples, num_samples=num_samples)
model_iid, trainer_iid = model_configure(p_iid_dataset, iters=iters, learn_rate=learn_rate)
print(p_iid_dataset.get_block_size)


NameError: name 'FixedModelDataset' is not defined

In [ ]:
# Train our model

train_run(model_iid, trainer_iid, 'iid_model.pt')

# Model for p=0.3

In [ ]:
p_03_dataset = FixedModelDataset(p=0.3, n=seq_samples, num_samples=num_samples)
model_p03, trainer_p03 = model_configure(p_03_dataset, iters=iters, learn_rate=learn_rate)

number of parameters: 0.11M
running on device cpu


In [ ]:
train_run(model_p03, trainer_p03, 'p03_model.pt')

Saving to ../models/p03_model.pt
iter_dt 0; iter 0: train loss 0.69202
iter_dt 100; iter 100: train loss 0.60875
iter_dt 200; iter 200: train loss 0.61003
iter_dt 300; iter 300: train loss 0.60998
iter_dt 400; iter 400: train loss 0.61365
iter_dt 500; iter 500: train loss 0.60845
iter_dt 600; iter 600: train loss 0.60997
iter_dt 700; iter 700: train loss 0.61144
iter_dt 800; iter 800: train loss 0.61074
iter_dt 900; iter 900: train loss 0.60768
Saved to ../models/p03_model.pt


# Model for p=0.7

In [ ]:
p_07_dataset = FixedModelDataset(p=0.7, n=seq_samples, num_samples=num_samples)
model_p07, trainer_p07 = model_configure(p_07_dataset, iters=iters, learn_rate=learn_rate)

number of parameters: 0.11M
running on device cpu


In [ ]:
train_run(model_p07, trainer_p07, 'p07_model.pt')

Saving to ../models/p07_model.pt
iter_dt 0; iter 0: train loss 0.68748
iter_dt 100; iter 100: train loss 0.61440
iter_dt 200; iter 200: train loss 0.61001
iter_dt 300; iter 300: train loss 0.61381
iter_dt 400; iter 400: train loss 0.60851
iter_dt 500; iter 500: train loss 0.60976
iter_dt 600; iter 600: train loss 0.60700
iter_dt 700; iter 700: train loss 0.60512
iter_dt 800; iter 800: train loss 0.61192
iter_dt 900; iter 900: train loss 0.61182
Saved to ../models/p07_model.pt


# Model for p = [0.1, 0.3, 0.5, 0.7, 0.9]

In [ ]:
p_points_dataset = FixedModelDataset(p=[0.1, 0.3, 0.5, 0.7, 0.9], n=seq_samples, num_samples=num_samples)
model_points, trainer_points = model_configure(p_points_dataset, iters=iters, learn_rate=learn_rate)

number of parameters: 0.11M
running on device cpu


In [ ]:
train_run(model_points, trainer_points, 'points_model.pt')

Saving to ../models/points_model.pt
iter_dt 0; iter 0: train loss 0.71105
iter_dt 100; iter 100: train loss 0.52107
iter_dt 200; iter 200: train loss 0.54795
iter_dt 300; iter 300: train loss 0.50006
iter_dt 400; iter 400: train loss 0.52561
iter_dt 500; iter 500: train loss 0.54379
iter_dt 600; iter 600: train loss 0.49400
iter_dt 700; iter 700: train loss 0.53954
iter_dt 800; iter 800: train loss 0.49276
iter_dt 900; iter 900: train loss 0.53400
Saved to ../models/points_model.pt
